This is the notebook to define the 8 Qubit QAOA in Bloqade SDK using python:

We need Hc, which is the 8 values that come out of H function after passing it the feateures variables

We need Hm, which we define it, a common way to define it is using RX gate (rotations in X)

We need to define the QAOA algorithm, that in essence is: 
- Starting the 8 qubits
- Apply Hadamard gates to create equal superposition
- Apply Hc part of the circuit which would be quantum gates 
- Apply Hm part of the circuit which would be quantum gates
- Measure it   

** After measurement you get a bitstring **

After having your QAOA defined, we also have to define auxiliar functions in python to make x quantity of shots per circuit, define the probability of a bitstring, the function to calculate energy/cost of a bitstring, a classical optimizer to update parameters from QAOA, run the circuit again, so we need the function to run the circuit

**pip installs**

In [1]:
pip install -q bloqade 

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
!{sys.executable} -m pip install -U lark

Install simulator

In [3]:
import sys
!{sys.executable} -m pip install -U bloqade
!{sys.executable} -m pip install -U "bloqade-circuit[tsim]"

  Using cached bloqade-0.32.0-py3-none-any.whl.metadata (2.9 kB)
INFO: pip is looking at multiple versions of bloqade to determine which version is compatible with other requirements. This could take a while.
  Using cached bloqade-0.31.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached bloqade-0.30.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached bloqade-0.29.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached bloqade-0.29.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached bloqade-0.28.3-py3-none-any.whl.metadata (2.9 kB)
  Using cached bloqade-0.28.2-py3-none-any.whl.metadata (2.9 kB)
  Using cached bloqade-0.28.1-py3-none-any.whl.metadata (2.9 kB)
INFO: pip is still looking at multiple versions of bloqade to determine which version is compatible with other requirements. This could take a while.
  Using cached bloqade-0.28.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached bloqade-0.27.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached bloqade-0.27.0-py3-none-any.whl.metadata 

  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [31 lines of output]
      Checking for Rust toolchain....
      Rust not found, installing into a temporary directory
      Python reports SOABI: cp314-win_amd64
      Computed rustc target triple: x86_64-pc-windows-msvc
      Installation directory: C:\Users\user\AppData\Local\puccinialin\puccinialin\Cache
      
      Installing rust to C:\Users\user\AppData\Local\puccinialin\puccinialin\Cache\rustup
      warn: It looks like you have an existing rustup settings file at:
      warn: C:\Users\user\.rustup\settings.toml
      warn: Rustup will install the default toolchain as specified in the settings file,
      warn: instead of the one inferred from the default host triple.
      warn: installing msvc toolchain without its prerequisites
      info: profile set to minimal
      info: default host triple is x86_64-pc-windows-msvc
      info: syncing channel

**Imports**

In [5]:
import math
from typing import Any
import numpy as np 
import kirin
from kirin.dialects import ilist

from bloqade import qasm2

pi = math.pi

from bloqade import squin, tsim
from bloqade.types import MeasurementResult, Qubit
from kirin.dialects.ilist import IList

Register = IList[Qubit, Any]
Measurement = IList[MeasurementResult, Any]

print("squin/tsim imports worked")

AttributeError: module 'kirin.types' has no attribute 'Complex'

**Global variables from Hamiltonian function we produced**

With these variables produced already, we do not need to use the dataset again or touch it in this ipynb because we already produced the hamiltonian cost on the variables we are using.

The ground truth was produced by brute forcing the 256 combinations and checking the best one, since is easy for a classical computer to do that 

In [ ]:
n_qubits = 8

qubit_labels = [
    "A004_LSB", "A004_MSB",
    "A047_LSB", "A047_MSB",
    "A026_LSB", "A026_MSB",
    "A017_LSB", "A017_MSB",
]

# Constant offset is irrelevant for QAOA optimization because it does not change
# which bitstring minimizes the objective.
c0 = 0.0

# FIXED H terms
# Local fields h_i for H_C = sum_i h_i Z_i + sum_{i<j} J_ij Z_i Z_j
h_terms = [
    (0, -0.019730),
    (1, -0.039461),
    (2, -0.014721),
    (3, -0.029441),
    (4, -0.028659),
    (5, -0.057318),
    (6, -0.047267),
    (7, -0.094534),
]

# FIXED J terms
J_terms = [
    (0, 1, 0.007132),
    (0, 2, 0.002667),
    (0, 3, 0.005334),
    (0, 4, 0.005334),
    (0, 5, 0.010667),
    (0, 6, 0.008889),
    (0, 7, 0.017778),
    (1, 2, 0.005334),
    (1, 3, 0.010667),
    (1, 4, 0.010667),
    (1, 5, 0.021335),
    (1, 6, 0.017778),
    (1, 7, 0.035555),
    (2, 3, 0.004005),
    (2, 4, 0.004000),
    (2, 5, 0.008000),
    (2, 6, 0.006667),
    (2, 7, 0.013335),
    (3, 4, 0.008000),
    (3, 5, 0.016000),
    (3, 6, 0.013335),
    (3, 7, 0.026665),
    (4, 5, 0.016005),
    (4, 6, 0.013335),
    (4, 7, 0.026667),
    (5, 6, 0.026667),
    (5, 7, 0.053335),
    (6, 7, 0.044450),
]


# Remains the same even after fix
ground_truth_bitstring = "11011110"


**First we are going to convert the Hamiltonian data to arrays**

In [ ]:
import numpy as np

# Build dense h vector and J matrix from the term lists
h8 = np.zeros(n_qubits, dtype=float)
J8 = np.zeros((n_qubits, n_qubits), dtype=float)

for i, hi in h_terms:
    h8[i] = hi

for i, j, Jij in J_terms:
    J8[i, j] = Jij
    J8[j, i] = Jij

ising_offset = 0.0

print("h8:")
print(h8)
print("\nJ8:")
print(J8)

**Classical Icing Energy**

In [ ]:
def ising_energy_from_x_list(x_list, h, J, ising_offset=0.0):
    # Use the convention x = (1 + s)/2, so s = 2x - 1
    s = np.array([2 * int(bit) - 1 for bit in x_list], dtype=float)

    energy = ising_offset

    for i in range(len(h)):
        energy += h[i] * s[i]

    for i in range(J.shape[0]):
        for j in range(i + 1, J.shape[1]):
            energy += J[i, j] * s[i] * s[j]

    return float(energy)

**Helper to convert bitstring to x-list**

In [ ]:
def bitstring_to_x_list(bitstring):
    return [int(b) for b in bitstring]

**Decode 2 bits per asset**

In [ ]:
def decode_lots_from_bitstring(bitstring):
    """
    Decode 8 qubits into 4 asset lot values using:
    lot = LSB + 2 * MSB
    """
    bits = [int(b) for b in bitstring]
    lots = []

    for k in range(0, len(bits), 2):
        lsb = bits[k]
        msb = bits[k + 1]
        lot = lsb + 2 * msb
        lots.append(lot)

    return lots

**Verify the portfolio**

In [ ]:
decoded_lots = decode_lots_from_bitstring(ground_truth_bitstring)
print("Decoded lots from ground-truth bitstring:", decoded_lots)

**Test the ground truth**

In [ ]:
ground_truth_x = [1, 1, 0, 1, 1, 1, 1, 0]

print("Ground-truth x:", ground_truth_x)
print("Ground-truth lots:", decode_lots_from_bitstring("".join(str(b) for b in ground_truth_x)))
print("Ground-truth corrected Ising energy:", ising_energy_from_x_list(ground_truth_x, h8, J8, ising_offset))

**Brute-force classical check over all 256 bitstrings**

In [ ]:
def x_list_to_lots(x_list):
    lots = []
    for k in range(0, len(x_list), 2):
        lsb = x_list[k]
        msb = x_list[k + 1]
        lot = lsb + 2 * msb
        lots.append(lot)
    return lots

all_results_rebuilt = []

for num in range(2 ** n_qubits):
    x_list = [(num >> i) & 1 for i in range(n_qubits)]
    e_ising = ising_energy_from_x_list(x_list, h8, J8, ising_offset)

    all_results_rebuilt.append({
        "x": x_list,
        "lots": x_list_to_lots(x_list),
        "ising_energy": e_ising
    })

all_results_rebuilt = sorted(all_results_rebuilt, key=lambda r: r["ising_energy"])

print("Top 15 solutions ranked by corrected Ising energy:\n")
for rank, r in enumerate(all_results_rebuilt[:15], start=1):
    print(f"{rank:2d}. x={r['x']}  lots={r['lots']}  Ising={r['ising_energy']:.8f}")

**Check the rank of the known ground truth**

In [ ]:
gt_rank = None

for idx, r in enumerate(all_results_rebuilt):
    if r["x"] == ground_truth_x:
        gt_rank = idx + 1
        print("Ground-truth entry found:")
        print(r)
        break

print("\nGround-truth rank under corrected Ising evaluation:", gt_rank)

# We actually create QAOA now as a bloqade quantum circuit

Import the simulator


In [ ]:
from bloqade.pyqrack import StackMemorySimulator

**Build the QAOA algorithm on Bloqade**


In [ ]:
def build_qaoa_ising_kernel(n_qubits, h_terms, J_terms):
    @qasm2.extended
    def kernel(gamma: ilist.IList[float, Any], beta: ilist.IList[float, Any]):
        q = qasm2.qreg(n_qubits)

        # Initialize the register in the |+> state
        for i in range(n_qubits):
            qasm2.h(q[i])

        # Apply p QAOA layers
        for layer in range(len(gamma)):
            g = gamma[layer]
            b = beta[layer]

            # Single-qubit Z terms
            for k in range(len(h_terms)):
                term = h_terms[k]
                i = term[0]
                hi = term[1]
                qasm2.rz(q[i], 2.0 * g * hi)

            # Two-qubit ZZ terms
            for k in range(len(J_terms)):
                term = J_terms[k]
                i = term[0]
                j = term[1]
                Jij = term[2]

                qasm2.cx(q[i], q[j])
                qasm2.rz(q[j], 2.0 * g * Jij)
                qasm2.cx(q[i], q[j])

            # Mixer layer
            for i in range(n_qubits):
                qasm2.rx(q[i], 2.0 * b)

        return q

    return kernel

**Create the Kernel**

In [ ]:
kernel = build_qaoa_ising_kernel(n_qubits, h_terms, J_terms)
print(kernel)

In [ ]:
print(kernel)
print(type(kernel))

**Choose initial QAOA parameters**

In [ ]:
# P stands for the depth of the circuit, how many times we repeat the gates
p = 1

gamma_init = [0.3] * p
beta_init = [0.2] * p

**wrap in main and emit QASM**

In [ ]:
@qasm2.extended
def main():
    return kernel(gamma_init, beta_init)

**print the generated circuit**

In [ ]:
target = qasm2.emit.QASM2()
ast = target.emit(main)
qasm2.parse.pprint(ast)

**run the circuit and get the state vector**

In [ ]:
sim = StackMemorySimulator(min_qubits=n_qubits)

ket = sim.state_vector(main)

print("State vector length:", len(ket))
print("Number of qubits:", n_qubits)

**Convert the state vector into probabilities**

In [ ]:
probs = np.abs(np.array(ket))**2

print("Probability sum:", probs.sum())

**helper to show the most likely bitstrings**

In [ ]:
def index_to_bitstring(idx, n):
    return format(idx, f"0{n}b")

top_k = 15
top_indices = np.argsort(probs)[::-1][:top_k]

print(f"Top {top_k} most likely computational basis states:\n")
for rank, idx in enumerate(top_indices, start=1):
    bitstring = index_to_bitstring(idx, n_qubits)
    prob = probs[idx]
    print(f"{rank:2d}. bitstring={bitstring}   prob={prob:.8f}")

**check the ground-truth probability**

In [ ]:
ground_truth_bitstring = "11011110"
gt_index = int(ground_truth_bitstring, 2)

print("Ground-truth bitstring:", ground_truth_bitstring)
print("Ground-truth probability:", probs[gt_index])

**Compute ising energy for the most likely bitstrings**

In [ ]:
def bitstring_to_x_list(bitstring):
    return [int(b) for b in bitstring]

print("Top states with Ising energy:\n")
for rank, idx in enumerate(top_indices, start=1):
    bitstring = index_to_bitstring(idx, n_qubits)
    x_list = bitstring_to_x_list(bitstring)
    energy = ising_energy_from_x_list(x_list, h8, J8, ising_offset)
    print(f"{rank:2d}. bitstring={bitstring}   prob={probs[idx]:.8f}   energy={energy:.8f}")

NO LE ENTIENDO ES CHAT: Cell 8 — compare direct and reversed bitstring conventions

In [ ]:
ground_truth_x = [1, 1, 0, 1, 1, 1, 1, 0]

gt_direct = "".join(str(b) for b in ground_truth_x)
gt_reversed = gt_direct[::-1]

print("Direct ground-truth bitstring:  ", gt_direct, "prob =", probs[int(gt_direct, 2)])
print("Reversed ground-truth bitstring:", gt_reversed, "prob =", probs[int(gt_reversed, 2)])